In [0]:
dbutils.widgets.text("catalog_name", "haredecodes")
dbutils.widgets.text("storage_account", "haredecodesnew")
dbutils.widgets.text("container_name", "data")
dbutils.widgets.text("raw_path_prefix", "staging")

catalog = dbutils.widgets.get("catalog_name")
storage = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container_name")
raw_prefix = dbutils.widgets.get("raw_path_prefix")

base = f"abfss://{container}@{storage}.dfs.core.windows.net"


In [0]:
spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {catalog}.bronze""")


In [0]:

source_path = f"{base}/{raw_prefix}/patients/"
checkpoint_path = f"{base}/bronze/patient_raw/checkpoint/"
schema_location = f"{base}/bronze/patient_raw/schema/"

df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("cloudFiles.maxFilesPerTrigger", 1)
    .option("cloudFiles.schemaLocation", schema_location)
    .load(source_path)
)

(
    df.drop("_rescued_data")
    .writeStream.format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(f"{catalog}.bronze.patient_raw")
)


In [0]:
spark.sql(f"""SELECT * FROM {catalog}.bronze.patient_raw""")
